### Determining Sample Size: Precision vs. Resources

Determining the appropriate sample size ($n$) is a fundamental trade-off in research between achieving desired precision and minimizing resource expenditure.

#### 1. The Core Objective

We aim to find the minimum $n$ that satisfies a specific **Margin of Error ($E$)** and **Confidence Level ($1 - \alpha$)**.

- **Small $n$:** Leads to high variance and increased risk of Type II errors (failing to detect a real effect).
- **Large $n$:** Increases precision but incurs higher costs (time, money).

#### 2. Sample Size for a Population Mean ($\mu$)

Derived from the $Z$-interval margin of error formula ($E = Z_{\alpha/2} \cdot \frac{\sigma}{\sqrt{n}}$), the required sample size is:

$$ n = \left( \frac{Z_{\alpha/2} \cdot \sigma}{E} \right)^2 $$

**Key Inputs:**
- $Z_{\alpha/2}$: Critical value based on the confidence level.
- $\sigma$: Population standard deviation.
- $E$: Tolerable margin of error.

> **Note:** Because calculated values are rarely integers, always **round up** to the next whole number to ensure sufficient statistical power.

In [ ]:
# ---------------------------------------------------------
# STEP 1: CALCULATING SAMPLE SIZE FOR A POPULATION MEAN
# ---------------------------------------------------------
import math
import scipy.stats as stats

def calculate_n_mean(confidence_level, population_std, margin_of_error):
    """
    Calculates the minimum required sample size for estimating a population mean.
    
    Parameters:
    confidence_level (float): The desired confidence level (e.g., 0.95 for 95%)
    population_std (float): The estimated population standard deviation (sigma)
    margin_of_error (float): The maximum acceptable margin of error (E)
    
    Returns:
    int: The rounded-up minimum sample size.
    """
    # Calculate alpha from the confidence level
    alpha = 1.0 - confidence_level
    
    # Calculate the critical Z value (two-tailed)
    # For a 95% confidence level, this will be approximately 1.96
    z_critical = stats.norm.ppf(1.0 - (alpha / 2.0))
    
    # Apply the mathematical formula: n = ( (Z * sigma) / E )^2
    n_raw = ((z_critical * population_std) / margin_of_error) ** 2
    
    # Always use the ceiling function to round up to the next whole person/item
    n_final = math.ceil(n_raw)
    
    print(f"--- Sample Size Calculation for Mean ---")
    print(f"Confidence Level : {confidence_level*100}%")
    print(f"Z-Critical Value : {z_critical:.4f}")
    print(f"Standard Dev (σ) : {population_std}")
    print(f"Margin Error (E) : {margin_of_error}")
    print(f"Raw Calculation  : {n_raw:.4f}")
    print(f"Required Size (n): {n_final}\n")
    
    return n_final

# Example Execution:
# Imagine testing the breaking strength of a new composite material.
# We estimate std_dev = 15 kg, want a margin of error of 2 kg, at 99% confidence.
required_n = calculate_n_mean(confidence_level=0.99, 
                              population_std=15.0, 
                              margin_of_error=2.0)

#### How to estimate $\sigma$ (the standard deviation)

Often, the true population standard deviation $\sigma$ is unknown. We can estimate it using:
- **Pilot Study:** Perform a small initial run to calculate sample standard deviation ($s$) as a proxy.
- **Historical Data:** Utilize findings from previous, similar research.
- **Range Rule of Thumb:** Estimate $\sigma \approx \frac{\text{Maximum} - \text{Minimum}}{4}$.

In [ ]:
# ---------------------------------------------------------
# STEP 2: ESTIMATING STANDARD DEVIATION VIA PILOT STUDY & RULE OF THUMB
# ---------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Seed for reproducibility
np.random.seed(42)

# Simulate a small pilot study (n=30) of manufacturing defect sizes (in mm)
pilot_data = np.random.normal(loc=50.0, scale=8.5, size=30)

# 1. Estimate using the Sample Standard Deviation (s)
s_proxy = np.std(pilot_data, ddof=1) # ddof=1 for sample standard deviation

# 2. Estimate using the Range Rule of Thumb (Max - Min) / 4
data_max = np.max(pilot_data)
data_min = np.min(pilot_data)
range_rule_proxy = (data_max - data_min) / 4.0

print("--- Estimating Unknown Population Variance ---")
print(f"Pilot Sample Std Dev (s): {s_proxy:.4f}")
print(f"Range Rule of Thumb (σ):  {range_rule_proxy:.4f}\n")

# Visualizing the Pilot Data to confirm distribution shape
plt.figure(figsize=(10, 5))
sns.histplot(pilot_data, kde=True, bins=8, color='steelblue', alpha=0.7)
plt.axvline(np.mean(pilot_data), color='red', linestyle='--', label=f'Mean: {np.mean(pilot_data):.2f}')
plt.title('Pilot Study Distribution: Assessing Variance Before Full Scaling', fontsize=14, pad=15)
plt.xlabel('Measurement Value (mm)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

#### 3. Sample Size for a Population Proportion ($p$)

When researching categorical data (e.g., "Yes/No" or "Pass/Fail"), we measure proportions instead of means. The formula is:

$$ n = p(1 - p) \left( \frac{Z_{\alpha/2}}{E} \right)^2 $$

#### The "Conservative Estimate" Strategy

If the population proportion ($p$) is completely unknown, you should use the most conservative estimate to ensure the sample is strictly large enough. 
The quadratic product $p(1-p)$ reaches its absolute maximum peak at $p = 0.5$.

**Conservative Formula:**
$$ n = 0.25 \left( \frac{Z_{\alpha/2}}{E} \right)^2 $$

In [ ]:
# ---------------------------------------------------------
# STEP 3: CALCULATING SAMPLE SIZE FOR A PROPORTION
# ---------------------------------------------------------

def calculate_n_proportion(confidence_level, estimated_p, margin_of_error):
    """
    Calculates the minimum sample size for estimating a population proportion.
    If estimated_p is None, it defaults to the conservative 0.5 estimate.
    """
    alpha = 1.0 - confidence_level
    z_critical = stats.norm.ppf(1.0 - (alpha / 2.0))
    
    # Apply conservative strategy if historical proportion is unknown
    if estimated_p is None:
        print("No prior proportion provided. Using conservative estimate (p=0.5).")
        p = 0.5
    else:
        p = estimated_p
        
    # Calculate variance factor p(1-p)
    variance_factor = p * (1.0 - p)
    
    # Apply the mathematical formula: n = p(1-p) * (Z / E)^2
    n_raw = variance_factor * ((z_critical / margin_of_error) ** 2)
    
    # Always round up to ensure we do not fall under the required threshold
    n_final = math.ceil(n_raw)
    
    print(f"--- Sample Size Calculation for Proportion ---")
    print(f"Confidence Level : {confidence_level*100}%")
    print(f"Z-Critical Value : {z_critical:.4f}")
    print(f"Estimated p      : {p}")
    print(f"Margin Error (E) : {margin_of_error}")
    print(f"Required Size (n): {n_final}\n")
    
    return n_final

# Example 1: With prior knowledge (e.g., historical defect rate is 15%)
n_prior = calculate_n_proportion(0.95, 0.15, 0.04)

# Example 2: Completely blind study (Conservative method)
n_blind = calculate_n_proportion(0.95, None, 0.04)

In [ ]:
# ---------------------------------------------------------
# STEP 4: VISUALIZING THE CONSERVATIVE PROPORTION RULE
# ---------------------------------------------------------
# To prove why 0.5 is the safest bet, we plot p(1-p) for all p between 0 and 1

# Generate an array of possible proportions from 0.0 to 1.0
p_values = np.linspace(0.0, 1.0, 100)

# Calculate the variance factor p(1-p) for each possible proportion
variance_factors = p_values * (1 - p_values)

plt.figure(figsize=(10, 5))

# Plot the parabola
plt.plot(p_values, variance_factors, color='darkorange', linewidth=3, label='$p(1-p)$ curve')

# Highlight the maximum point
plt.axvline(0.5, color='red', linestyle='--', alpha=0.6)
plt.axhline(0.25, color='red', linestyle='--', alpha=0.6)
plt.scatter(0.5, 0.25, color='red', s=100, zorder=5, label='Maximum Variance (0.25) at p=0.5')

# Formatting the chart for readability
plt.title('Why We Use p=0.5 for Conservative Sample Size Estimates', fontsize=14, pad=15)
plt.xlabel('True Population Proportion ($p$)', fontsize=12)
plt.ylabel('Variance Factor $p(1-p)$', fontsize=12)
plt.xlim(0, 1)
plt.ylim(0, 0.3)
plt.legend(loc='lower center')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### 4. Summary Table of Inputs

Understanding how tuning these parameters affects your required sample size is critical for experimental design.

| **Input Parameter** | **Description** | **Impact of Increase on Sample Size ($n$)** |
| :--- | :--- | :--- |
| **Margin of Error ($E$)** | Tolerated deviation from true parameter | Higher $E \rightarrow$ **Smaller $n$** (Inverse relationship) |
| **Confidence Level** | Desired certainty (e.g., 90% vs 99%) | Higher Confidence $\rightarrow$ **Larger $n$** |
| **Variability ($\sigma$ or $p$)** | Heterogeneity in the population | Higher Variance $\rightarrow$ **Larger $n$** |

In [ ]:
# ---------------------------------------------------------
# STEP 5: SENSITIVITY ANALYSIS (HEATMAP & CURVES)
# ---------------------------------------------------------
# Let's visualize the trade-offs dynamically across multiple Margins of Error
# and Confidence Levels to show the exponential cost of precision.

# Define parameters for sensitivity analysis
margins = np.linspace(0.01, 0.10, 10)     # Margin of errors from 1% to 10%
conf_levels = [0.90, 0.95, 0.99]          # Standard confidence levels
fixed_p = 0.5                             # Worst case proportion scenario

# Initialize a dictionary to store arrays of required n
results_dict = {}

# Calculate sample sizes for every combination
for conf in conf_levels:
    n_array = []
    alpha = 1.0 - conf
    z_crit = stats.norm.ppf(1.0 - (alpha / 2.0))
    
    for e in margins:
        # n = p(1-p) * (Z/E)^2
        n = math.ceil((fixed_p * (1 - fixed_p)) * ((z_crit / e) ** 2))
        n_array.append(n)
        
    results_dict[f"{int(conf*100)}%"] = n_array

# Convert to DataFrame for easy seaborn plotting
df_sensitivity = pd.DataFrame(results_dict, index=np.round(margins, 3))

# Set up the Matplotlib Figure with 2 Subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Subplot 1: Line Plot (Exponential Curve) ---
for col in df_sensitivity.columns:
    axes[0].plot(df_sensitivity.index, df_sensitivity[col], marker='o', linewidth=2, label=f'{col} Confidence')

axes[0].set_title('Sample Size vs. Margin of Error', fontsize=14, pad=10)
axes[0].set_xlabel('Margin of Error ($E$)', fontsize=12)
axes[0].set_ylabel('Required Sample Size ($n$)', fontsize=12)
axes[0].invert_xaxis() # Invert to show increasing precision from left to right
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# --- Subplot 2: Heatmap (Decision Matrix) ---
# Transpose for better visual orientation in the heatmap
sns.heatmap(df_sensitivity.T, annot=True, fmt="d", cmap="YlGnBu", ax=axes[1],
            cbar_kws={'label': 'Required Sample Size ($n$)'})

axes[1].set_title('Decision Matrix: Resource Trade-offs', fontsize=14, pad=10)
axes[1].set_xlabel('Margin of Error ($E$)', fontsize=12)
axes[1].set_ylabel('Confidence Level', fontsize=12)

plt.tight_layout()
plt.show()

# Print out the final dataset confirming the exact exponential scaling
print("\nTabular Results of Sensitivity Analysis:")
display(df_sensitivity.rename_axis("Margin of Error (E)"))